# DENDRA for reinforcement learning: Atari (discrete actions) and MuJoCo (continuous actions)

Compares 3 "cores" inside the same recurrent PPO agent:

| Core | Memory |
| --- | --- |
| **DENDRA** | Hebbian synapses + spiking E/I neurons + neuromodulators |
| **LSTM** | memory cells (the standard for recurrent RL) |
| **Memoryless** | CNN/MLP that sees the last 4 frames (the standard for Atari) |

**Default environments**
* `atari:Breakout`: real Atari 2600, 84x84 images, discrete actions.
* `mujoco:HalfCheetah-v5-pomdp`: a running robot, 6-D continuous actions, **velocities hidden** (only joint positions are visible), so the agent must remember to infer them.

**How to run:** Accelerator **GPU T4 x2**, turn on **Internet**, *Run All*. You can use *Save Version -> Save & Run All* to run in the background (up to ~12 hours).
Each run has a time limit `MAX_MINUTES`; results are still saved if a run stops early.

In [ ]:
!pip install -q "gymnasium>=1.1" ale-py mujoco minatar
import gymnasium, ale_py, mujoco, torch, os
print("gymnasium", gymnasium.__version__, "| ale-py", ale_py.__version__, "| mujoco", mujoco.__version__,
      "| torch", torch.__version__, "| GPU:", torch.cuda.device_count())

In [ ]:
# Write the project source files to disk
import os
FILES = {'dendra/__init__.py': 'from .model import DENDRA, GPT, HebbianSTDP, DendriticSpikingLayer, CorticalRegion, Neuromodulators\n', 'dendra/model.py': '"""\nDENDRA: a language model built from biological neuron mechanisms.\n\nMain components\n  LIF neuron + refractory period -> DendriticSpikingLayer (0/1 spikes, soft reset, adaptive threshold)\n  Nonlinear dendrites            -> each neuron has B branches, each with its own local nonlinearity\n  E (80%) / I (20%) populations  -> Dale\'s law: E->I and I->E weights are positive; I subtracts from E (soft k-WTA)\n  Hebb + STDP + NMDA             -> HebbianSTDP: S <- lam*S + w * v k^T  (replaces attention)\n  Neuromodulators (NE, ACh)      -> Neuromodulators: global gain and memory-write gate\n  Feedback connections (L6)      -> several up/down passes, higher regions send signals down\n  Spinal reflex                  -> early-exit head after region 1\n  Homeostasis                    -> loss keeping the firing rate near a target (in the training script)\n\nThree computation modes for the same weights (outputs match, max error < 1e-5):\n  forward()      parallel over time, used for training\n  prefill()      reads a prompt chunk by chunk, O(n * C)\n  step_token()   generates one token with a constant-size state, O(1) memory per token\n"""\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n# ---------------------------------------------------------------- spikes + surrogate gradient\nclass _Spike(torch.autograd.Function):\n    """All-or-none: spike = 1 when the membrane potential crosses threshold. Surrogate gradient: fast sigmoid."""\n\n    @staticmethod\n    def forward(ctx, x, alpha):\n        ctx.save_for_backward(x)\n        ctx.alpha = alpha\n        return (x > 0).to(x.dtype)\n\n    @staticmethod\n    def backward(ctx, g):\n        (x,) = ctx.saved_tensors\n        return g / (1.0 + ctx.alpha * x.abs()) ** 2, None\n\n\ndef spike(x, alpha=2.0):\n    return _Spike.apply(x, alpha)\n\n\nclass DivisiveNorm(nn.Module):\n    """Divisive normalization by an interneuron pool: divide by the mean activity energy."""\n\n    def __init__(self, d, eps=1e-5):\n        super().__init__()\n        self.g = nn.Parameter(torch.ones(d))\n        self.eps = eps\n\n    def forward(self, x):\n        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.g\n\n\n# ---------------------------------------------------------------- L2/3: Hebbian/STDP synapses\nclass HebbianSTDP(nn.Module):\n    """\n    Fast-weight memory that replaces attention.\n      S_t = lam * S_{t-1} + w_t * v_t k_t^T        (Hebb: pre and post active together -> stronger synapse)\n      o_t = S_t q_t\n    - lam < 1: synaptic decay; each head has its own time constant (4..256 tokens), learned.\n    - Parallel form: o = ((Q K^T) * D * w) V with D_ij = lam^(i-j) for j <= i. This is a causal STDP window\n      (pre-before-post strengthens, decaying with distance), so no position embedding is needed.\n    - w_t = NMDA(x_t) * ACh_t: write only when the postsynaptic neuron is depolarized (NMDA gate) and ACh is high.\n    """\n\n    def __init__(self, d, n_heads, tau_min=4.0, tau_max=256.0):\n        super().__init__()\n        assert d % n_heads == 0\n        self.h, self.dh = n_heads, d // n_heads\n        self.qkv = nn.Linear(d, 3 * d, bias=False)\n        self.nmda = nn.Linear(d, n_heads)  # coincidence-detector gate\n        self.og = nn.Linear(d, d, bias=False)  # output gate (postsynaptic depolarization)\n        self.out = nn.Linear(d, d, bias=False)\n        self.onorm = DivisiveNorm(self.dh)\n        tau = torch.exp(torch.linspace(math.log(tau_min), math.log(tau_max), n_heads))\n        lam = 1.0 - 1.0 / tau\n        self.lam_logit = nn.Parameter(torch.log(lam / (1 - lam)))  # learnable\n        nn.init.constant_(self.nmda.bias, 2.0)\n\n    def lam(self):\n        return torch.sigmoid(self.lam_logit)  # [h]\n\n    def _proj(self, x, ach):\n        B, T, _ = x.shape\n        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).unbind(2)  # [B,T,h,dh]\n        q = q * self.dh ** -0.5\n        k = F.normalize(k, dim=-1)\n        w = torch.sigmoid(self.nmda(x))  # [B,T,h]\n        if ach is not None:\n            w = w * ach\n        return q, k, v, w\n\n    def forward(self, x, ach=None):\n        B, T, _ = x.shape\n        q, k, v, w = self._proj(x, ach)\n        idx = torch.arange(T, device=x.device)\n        dist = (idx[:, None] - idx[None, :]).float()  # i - j\n        causal = dist >= 0\n        loglam = torch.log(self.lam())[:, None, None]  # [h,1,1]\n        D = torch.where(causal, torch.exp(loglam * dist.clamp(min=0)), torch.zeros((), device=x.device))\n        A = torch.einsum("bihd,bjhd->bhij", q, k) * D  # [B,h,T,T]\n        A = A * w.permute(0, 2, 1)[:, :, None, :]  # write strength of each token j\n        o = torch.einsum("bhij,bjhd->bihd", A, v)\n        return self._finish(x, o)\n\n    def _finish(self, x, o):\n        B, T = x.shape[:2]\n        o = self.onorm(o).reshape(B, T, -1)\n        return self.out(o * F.silu(self.og(x)))\n\n    def chunked(self, x, ach, S, C=128):\n        """Read a sequence in chunks of C tokens: parallel inside a chunk, state S carried between chunks.\n        Same result as forward() but O(n*C) instead of O(n^2); also returns the final state."""\n        B, T, _ = x.shape\n        q, k, v, w = self._proj(x, ach)\n        lam = self.lam()  # [h]\n        loglam = torch.log(lam)\n        outs = []\n        for s0 in range(0, T, C):\n            qc, kc, vc, wc = q[:, s0:s0 + C], k[:, s0:s0 + C], v[:, s0:s0 + C], w[:, s0:s0 + C]\n            n = qc.shape[1]\n            idx = torch.arange(n, device=x.device)\n            dist = (idx[:, None] - idx[None, :]).float()\n            D = torch.where(dist >= 0, torch.exp(loglam[:, None, None] * dist.clamp(min=0)), torch.zeros((), device=x.device))\n            A = torch.einsum("bihd,bjhd->bhij", qc, kc) * D * wc.permute(0, 2, 1)[:, :, None, :]\n            o = torch.einsum("bhij,bjhd->bihd", A, vc)  # within-chunk part\n            decay_in = torch.exp(loglam[None, :] * (idx[:, None] + 1).float())  # lam^(a+1) [n,h]\n            o = o + torch.einsum("bhde,bihe->bihd", S.to(qc.dtype), qc) * decay_in[None, :, :, None].to(qc.dtype)\n            decay_out = torch.exp(loglam[None, :] * (n - 1 - idx)[:, None].float())  # lam^(n-1-b) [n,h]\n            wv = vc * (wc * decay_out[None].to(wc.dtype))[..., None]\n            S = lam[None, :, None, None] ** n * S + torch.einsum("bjhd,bjhe->bhde", wv.float(), kc.float())\n            outs.append(o)\n        return self._finish(x, torch.cat(outs, 1)), S\n\n    def step(self, x, ach, S):\n        """One-token inference: constant-size memory S [B,h,dh,dh], no KV cache."""\n        q, k, v, w = self._proj(x, ach)  # x [B,1,d]\n        q, k, v, w = q[:, 0], k[:, 0], v[:, 0], w[:, 0]\n        lam = self.lam()[None, :, None, None]\n        S = lam * S + w[..., None, None] * torch.einsum("bhd,bhe->bhde", v, k)\n        o = torch.einsum("bhde,bhe->bhd", S, q)[:, None]\n        return self._finish(x, o), S\n\n\n# ---------------------------------------------------------------- L5: spiking dendritic neurons\nclass DendriticSpikingLayer(nn.Module):\n    """\n    E population with dendrites + inhibitory I population, run for T gamma ticks per token.\n      branch input    c_b = W_b x                (synapses on each dendritic branch)\n      soma input      I = sum_b a_b * g(c_b)     (g: local branch nonlinearity, a_b >= 0)\n      E potential     u <- beta*u + gain*I - W_IE s_I\n      E spikes        s_E = H(u - (theta + adapt)); soft reset u -= theta*s_E; adapt <- rho*adapt + gamma*s_E\n      I population    ui <- beta_i*ui + W_EI s_E;  s_I = H(ui - theta_I)\n    Dale\'s law: W_EI, W_IE >= 0 (softplus). Output = E firing rate -> read out into the residual stream.\n    """\n\n    def __init__(self, d, n_e, branches=4, T=4, inh_ratio=0.25, theta=1.0, spike_in=False):\n        super().__init__()\n        self.spike_in = spike_in\n        if spike_in:  # encode the input as ON/OFF spikes (like retinal cells) -> dendritic synapses only add\n            self.theta_in_raw = nn.Parameter(torch.full((d,), 1.0))\n        self.ne, self.B, self.T = n_e, branches, T\n        self.ni = max(1, int(n_e * inh_ratio))  # inhibitory neurons are 20% of the total\n        self.dend = nn.Linear(d, n_e * branches)\n        self.branch_raw = nn.Parameter(torch.zeros(n_e, branches))  # softplus(0)=0.69\n        self.beta_raw = nn.Parameter(torch.full((n_e,), 1.5))  # membrane leak, sigmoid ~ 0.82\n        self.w_ei_raw = nn.Parameter(torch.randn(n_e, self.ni) * 0.1 - 4.0)\n        self.w_ie_raw = nn.Parameter(torch.randn(self.ni, n_e) * 0.1 - 3.0)\n        self.theta, self.theta_i = theta, 1.0\n        self.rho, self.gamma = 0.5, 0.5  # refractory period / threshold adaptation\n        self.out = nn.Linear(n_e, d, bias=False)\n        nn.init.normal_(self.out.weight, std=0.02)\n\n    def encode(self, x):\n        th = F.softplus(self.theta_in_raw) + 0.1\n        up, un = torch.zeros_like(x), torch.zeros_like(x)\n        cp, cn = torch.zeros_like(x), torch.zeros_like(x)\n        for _ in range(self.T):\n            up = up + x\n            un = un - x\n            sp, sn = spike(up - th), spike(un - th)\n            up, un = up - th * sp, un - th * sn\n            cp, cn = cp + sp, cn + sn\n        self.last_in_spikes = (cp + cn).sum(-1).mean().detach()  # input spikes per token\n        return (cp - cn) / self.T * th  # quantized value, same scale\n\n    def forward(self, x, gain=None):\n        Bt = x.shape[:-1]\n        if self.spike_in:\n            x = self.encode(x)\n        c = self.dend(x).view(*Bt, self.ne, self.B)\n        I = (F.softplus(self.branch_raw) * F.silu(c)).sum(-1)  # spatial integration over dendrites\n        if gain is not None:\n            I = I * gain\n        beta = torch.sigmoid(self.beta_raw)\n        w_ei = F.softplus(self.w_ei_raw)\n        w_ie = F.softplus(self.w_ie_raw)\n        u = torch.zeros_like(I)\n        adapt = torch.zeros_like(I)\n        ui = torch.zeros(*Bt, self.ni, device=x.device, dtype=x.dtype)\n        s_i = torch.zeros_like(ui)\n        count = torch.zeros_like(I)\n        count_i = torch.zeros_like(ui)\n        for _ in range(self.T):  # T gamma ticks\n            u = beta * u + I - s_i @ w_ie\n            s = spike(u - (self.theta + adapt))\n            u = u - self.theta * s\n            adapt = self.rho * adapt + self.gamma * s\n            ui = 0.5 * ui + s @ w_ei\n            s_i = spike(ui - self.theta_i)\n            ui = ui - self.theta_i * s_i\n            count = count + s\n            count_i = count_i + s_i\n        rate = count / self.T\n        self.last_rate = rate.mean()\n        self.last_rate_i = (count_i / self.T).mean().detach()\n        return self.out(rate)\n\n\n# ---------------------------------------------------------------- one cortical region (6-layer column)\nclass CorticalRegion(nn.Module):\n    """L4 input (divisive inhibition) -> L2/3 mixing over time -> L5 spiking output -> L6 receives feedback."""\n\n    def __init__(self, d, n_heads, n_e, branches, T, spike_in=False):\n        super().__init__()\n        self.l4a = DivisiveNorm(d)\n        self.l23 = HebbianSTDP(d, n_heads)\n        self.l4b = DivisiveNorm(d)\n        self.l5 = DendriticSpikingLayer(d, n_e, branches, T, spike_in=spike_in)\n        self.l6_norm = DivisiveNorm(d)\n        self.l6 = nn.Linear(d, d, bias=False)  # receives predictions from the region above\n        nn.init.zeros_(self.l6.weight)\n\n    def topdown(self, h_above):\n        return self.l6(self.l6_norm(h_above))\n\n    def forward(self, h, gain=None, ach=None, fb=None):\n        if fb is not None:\n            h = h + fb\n        h = h + self.l23(self.l4a(h), ach)\n        h = h + self.l5(self.l4b(h), gain)\n        return h\n\n    def forward_chunked(self, h, gain, ach, fb, S, C=128):\n        if fb is not None:\n            h = h + fb\n        o, S = self.l23.chunked(self.l4a(h), ach, S, C)\n        h = h + o\n        h = h + self.l5(self.l4b(h), gain)\n        return h, S\n\n    def step(self, h, gain, ach, fb, S):\n        if fb is not None:\n            h = h + fb\n        o, S = self.l23.step(self.l4a(h), ach, S)\n        h = h + o\n        h = h + self.l5(self.l4b(h), gain)\n        return h, S\n\n\nclass Neuromodulators(nn.Module):\n    """\'Brainstem\': from a baseline level (causal EMA of the input) emits noradrenaline (gain) and acetylcholine (write gate)."""\n\n    def __init__(self, d, n_regions, n_heads, decay=0.9):\n        super().__init__()\n        self.L, self.h, self.decay = n_regions, n_heads, decay\n        self.norm = DivisiveNorm(d)\n        self.proj = nn.Linear(d, n_regions * (1 + n_heads))\n        nn.init.zeros_(self.proj.weight)\n        nn.init.zeros_(self.proj.bias)  # start: gain = 1, ach = 1\n\n    def _split(self, m):\n        m = 2 * torch.sigmoid(self.proj(self.norm(m)))\n        m = m.view(*m.shape[:-1], self.L, 1 + self.h)\n        return m[..., 0:1], m[..., 1:]  # gain [..,L,1], ach [..,L,h]\n\n    def forward(self, x, K=256):\n        # causal EMA as a 1-D convolution (kernel truncated at K steps: decay^K ~ 1e-12), O(n*K) instead of O(n^2)\n        B, T, d = x.shape\n        K = min(K, T)\n        ker = (self.decay ** torch.arange(K - 1, -1, -1, device=x.device, dtype=torch.float32)).to(x.dtype)\n        xt = F.pad(x.transpose(1, 2), (K - 1, 0))\n        ema = F.conv1d(xt, ker.view(1, 1, K).expand(d, 1, K).contiguous(), groups=d).transpose(1, 2)\n        t = torch.arange(T, device=x.device, dtype=torch.float32)\n        norm = (1 - self.decay ** (t + 1).clamp(max=K)) / (1 - self.decay)\n        return self._split(ema / norm[None, :, None].to(ema.dtype))\n\n    def step(self, x, m_state, n_seen):\n        # normalized EMA, identical to the parallel form\n        m_state = self.decay * m_state + x\n        norm = (1 - self.decay ** (n_seen + 1)) / (1 - self.decay)\n        return self._split(m_state / norm), m_state\n\n\nclass DENDRA(nn.Module):\n    def __init__(self, vocab, d=128, n_regions=4, n_heads=4, n_e=192, branches=4, T=4,\n                 fb_cycles=1, use_neuromod=True, reflex=True, spike_in=False):\n        super().__init__()\n        self.emb = nn.Embedding(vocab, d)  # sensory encoding\n        nn.init.normal_(self.emb.weight, std=0.02)\n        self.regions = nn.ModuleList([CorticalRegion(d, n_heads, n_e, branches, T, spike_in) for _ in range(n_regions)])\n        self.nm = Neuromodulators(d, n_regions, n_heads) if use_neuromod else None\n        self.fb_cycles = fb_cycles\n        self.norm_out = DivisiveNorm(d)\n        self.reflex = reflex\n        if reflex:\n            self.norm_reflex = DivisiveNorm(d)\n        self.n_heads = n_heads\n\n    def _head(self, h, norm):\n        return norm(h) @ self.emb.weight.t()\n\n    def forward(self, idx):\n        x = self.emb(idx)\n        gain = ach = None\n        if self.nm is not None:\n            gain, ach = self.nm(x)\n        prev = None\n        for c in range(self.fb_cycles):\n            h, outs = x, []\n            for l, reg in enumerate(self.regions):\n                fb = None\n                if prev is not None and l + 1 < len(self.regions):\n                    fb = reg.topdown(prev[l + 1])  # prediction from the region above in the previous pass\n                h = reg(h, None if gain is None else gain[:, :, l], None if ach is None else ach[:, :, l], fb)\n                outs.append(h)\n            prev = outs\n        logits = self._head(h, self.norm_out)\n        reflex_logits = self._head(prev[0], self.norm_reflex) if self.reflex else None\n        return logits, reflex_logits\n\n    def spike_rates(self):\n        return [r.l5.last_rate for r in self.regions]\n\n    def init_state(self, B, dev):\n        d = self.emb.weight.shape[1]\n        L, h = len(self.regions), self.n_heads\n        dh = d // h\n        return {"S": [[torch.zeros(B, h, dh, dh, device=dev) for _ in range(L)] for _ in range(self.fb_cycles)],\n                "m": torch.zeros(B, 1, d, device=dev), "t": 0}\n\n    @torch.no_grad()\n    def prefill(self, idx, C=128):\n        """Read the whole prompt once (in chunks); return the last-token logits + state for further generation."""\n        B, T = idx.shape\n        st = self.init_state(B, idx.device)\n        x = self.emb(idx)\n        gain = ach = None\n        if self.nm is not None:\n            gain, ach = self.nm(x)\n            m = torch.zeros(B, 1, x.shape[-1], device=x.device, dtype=torch.float32)\n            dec = self.nm.decay ** torch.arange(T - 1, -1, -1, device=x.device, dtype=torch.float32)\n            st["m"] = (x.float() * dec[None, :, None]).sum(1, keepdim=True)  # unnormalized EMA sum\n        st["t"] = T\n        L = len(self.regions)\n        prev = None\n        for c in range(self.fb_cycles):\n            h, outs = x, []\n            for l, reg in enumerate(self.regions):\n                fb = reg.topdown(prev[l + 1]) if (prev is not None and l + 1 < L) else None\n                h, st["S"][c][l] = reg.forward_chunked(h, None if gain is None else gain[:, :, l],\n                                                       None if ach is None else ach[:, :, l], fb, st["S"][c][l], C)\n                outs.append(h)\n            prev = outs\n        return self._head(h[:, -1:], self.norm_out), st\n\n    @torch.no_grad()\n    def step_token(self, tok, st):\n        """Generate 1 token using only the constant-size synaptic state (O(1) memory per token)."""\n        L = len(self.regions)\n        x = self.emb(tok)\n        gain = ach = None\n        if self.nm is not None:\n            (gain, ach), st["m"] = self.nm.step(x, st["m"], st["t"])\n        st["t"] += 1\n        prev = None\n        for c in range(self.fb_cycles):\n            hcur, outs = x, []\n            for l, reg in enumerate(self.regions):\n                fb = reg.topdown(prev[l + 1]) if (prev is not None and l + 1 < L) else None\n                hcur, st["S"][c][l] = reg.step(hcur, None if gain is None else gain[:, :, l],\n                                               None if ach is None else ach[:, :, l], fb, st["S"][c][l])\n                outs.append(hcur)\n            prev = outs\n        return self._head(hcur, self.norm_out), st\n\n    @torch.no_grad()\n    def forward_recurrent(self, idx):\n        """Run token by token with the constant-size state (to check it matches the parallel forward)."""\n        st = self.init_state(idx.shape[0], idx.device)\n        outs = []\n        for t in range(idx.shape[1]):\n            lg, st = self.step_token(idx[:, t:t + 1], st)\n            outs.append(lg)\n        return torch.cat(outs, 1)\n\n\n# ---------------------------------------------------------------- GPT baseline (standard causal attention)\nclass GPTBlock(nn.Module):\n    def __init__(self, d, n_heads):\n        super().__init__()\n        self.h = n_heads\n        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)\n        self.qkv = nn.Linear(d, 3 * d, bias=False)\n        self.proj = nn.Linear(d, d, bias=False)\n        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))\n\n    def forward(self, x):\n        B, T, D = x.shape\n        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D)\n        x = x + self.proj(a)\n        return x + self.mlp(self.ln2(x))\n\n\nclass GPT(nn.Module):\n    def __init__(self, vocab, d=384, n_layers=6, n_heads=6, ctx=256):\n        super().__init__()\n        self.emb = nn.Embedding(vocab, d)\n        self.pos = nn.Embedding(ctx, d)\n        nn.init.normal_(self.emb.weight, std=0.02)\n        nn.init.normal_(self.pos.weight, std=0.02)\n        self.blocks = nn.ModuleList([GPTBlock(d, n_heads) for _ in range(n_layers)])\n        self.ln = nn.LayerNorm(d)\n\n    def forward(self, idx):\n        x = self.emb(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))\n        for b in self.blocks:\n            x = b(x)\n        return self.ln(x) @ self.emb.weight.t(), None\n\n    def _run(self, x, cache, causal):\n        for i, b in enumerate(self.blocks):\n            B, T, D = x.shape\n            q, k, v = b.qkv(b.ln1(x)).view(B, T, 3, b.h, D // b.h).permute(2, 0, 3, 1, 4)\n            if cache[i] is not None:\n                k = torch.cat([cache[i][0], k], 2); v = torch.cat([cache[i][1], v], 2)\n            cache[i] = (k, v)\n            a = F.scaled_dot_product_attention(q, k, v, is_causal=causal).transpose(1, 2).reshape(B, T, D)\n            x = x + b.proj(a)\n            x = x + b.mlp(b.ln2(x))\n        return self.ln(x) @ self.emb.weight.t(), cache\n\n    @torch.no_grad()\n    def prefill(self, idx):\n        """Read the prompt and store the KV cache (memory grows with context length)."""\n        x = self.emb(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))\n        lg, cache = self._run(x, [None] * len(self.blocks), True)\n        return lg[:, -1:], {"kv": cache, "t": idx.shape[1]}\n\n    @torch.no_grad()\n    def step_token(self, tok, st):\n        pos = min(st["t"], self.pos.num_embeddings - 1)\n        x = self.emb(tok) + self.pos(torch.tensor([pos], device=tok.device))\n        lg, st["kv"] = self._run(x, st["kv"], False)\n        st["t"] += 1\n        return lg, st\n', 'experiments/rl/ppo_recurrent.py': '"""\nRecurrent PPO for both DISCRETE actions (Atari, MinAtar) and CONTINUOUS actions (Pendulum, MuJoCo).\nSwappable core: DENDRA (Hebbian synapses + spiking neurons + neuromodulators), LSTM, or a memoryless MLP/CNN.\n\nEnvironments (--env):\n  atari:Breakout          real Atari 2600 (ALE), 84x84 grayscale, frame skip 4\n  minatar:breakout        miniature 10x10 Atari (MinAtar), fast on CPU\n  pendulum-pomdp          Pendulum with the angular velocity HIDDEN -> must be inferred from memory (continuous actions)\n  mujoco:HalfCheetah-v5   full MuJoCo; add the suffix "-pomdp" to keep positions only (velocities hidden)\n\nExample:\n  python experiments/rl/ppo_recurrent.py --env minatar:breakout --core dendra --total 300000 --out runs/ppo.json\n"""\nimport argparse, json, math, os, sys, time\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport gymnasium as gym\n\nsys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))\nfrom dendra.model import CorticalRegion, Neuromodulators\n\np = argparse.ArgumentParser()\np.add_argument("--env", default="minatar:breakout")\np.add_argument("--core", default="dendra", choices=["dendra", "lstm", "mlp"])\np.add_argument("--total", type=int, default=300_000, help="total environment steps")\np.add_argument("--num_envs", type=int, default=8)\np.add_argument("--num_steps", type=int, default=128)\np.add_argument("--minibatches", type=int, default=4)\np.add_argument("--epochs", type=int, default=None, help="default: 4 (discrete), 10 (continuous)")\np.add_argument("--lr", type=float, default=None)\np.add_argument("--d", type=int, default=128)\np.add_argument("--seed", type=int, default=0)\np.add_argument("--max_minutes", type=float, default=None, help="stop early (results are still saved) after this many minutes")\np.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\np.add_argument("--out", default="runs/ppo.json")\na = p.parse_args()\ntorch.manual_seed(a.seed); np.random.seed(a.seed)\ntorch.set_num_threads(int(os.environ.get("NT", "1")))\ndev = torch.device(a.device)\nKIND = a.env.split(":")[0]\nCONT = KIND in ("pendulum-pomdp", "mujoco")\n\n\n# ================================================================== environments\nclass MinAtarVec:\n    """Vectorized MinAtar, auto-resets at episode end. Observations [N, C, 10, 10]."""\n\n    def __init__(self, game, n, seed):\n        from minatar import Environment\n        self.envs = [Environment(game) for i in range(n)]\n        for i, e in enumerate(self.envs):\n            e.random = np.random.RandomState(seed * 1000 + i)   # independent random stream per environment\n            e.reset()\n        self.n_actions = self.envs[0].num_actions()\n        self.obs_shape = (self.envs[0].state_shape()[2], 10, 10)\n\n    def obs(self):\n        return np.stack([e.state().transpose(2, 0, 1) for e in self.envs]).astype(np.float32)\n\n    def reset(self):\n        return self.obs()\n\n    def step(self, actions):\n        rew, done = np.zeros(len(self.envs), np.float32), np.zeros(len(self.envs), np.float32)\n        for i, (e, act) in enumerate(zip(self.envs, actions)):\n            r, d = e.act(int(act))\n            rew[i], done[i] = r, float(d)\n            if d:\n                e.reset()\n        return self.obs(), rew, done\n\n\nclass GymVec:\n    """Gymnasium SyncVectorEnv that resets within the terminating step (SAME_STEP)."""\n\n    def __init__(self, thunk, n, seed):\n        self.v = gym.vector.SyncVectorEnv([thunk(i) for i in range(n)],\n                                          autoreset_mode=gym.vector.AutoresetMode.SAME_STEP)\n        self.seed = seed\n        sp = self.v.single_action_space\n        self.n_actions = sp.n if hasattr(sp, "n") else sp.shape[0]\n        self.obs_shape = self.v.single_observation_space.shape\n        if CONT:\n            self.low, self.high = sp.low, sp.high\n\n    def reset(self):\n        return self.v.reset(seed=self.seed)[0].astype(np.float32)\n\n    def step(self, actions):\n        if CONT:\n            actions = np.clip(actions, self.low, self.high)\n        o, r, term, trunc, _ = self.v.step(actions)\n        return o.astype(np.float32), r.astype(np.float32), (term | trunc).astype(np.float32)\n\n\ndef make_vec(stack):\n    if KIND == "minatar":\n        return MinAtarVec(a.env.split(":")[1], a.num_envs, a.seed)\n    if KIND == "atari":\n        import ale_py\n        gym.register_envs(ale_py)\n        game = a.env.split(":")[1]\n\n        def thunk(i):\n            def f():\n                e = gym.make(f"ALE/{game}-v5", frameskip=1)\n                e = gym.wrappers.AtariPreprocessing(e, frame_skip=4, screen_size=84, grayscale_obs=True, noop_max=30)\n                return gym.wrappers.FrameStackObservation(e, stack)\n            return f\n        return GymVec(thunk, a.num_envs, a.seed)\n    if KIND == "pendulum-pomdp":\n        def thunk(i):\n            def f():\n                e = gym.make("Pendulum-v1")\n                e = gym.wrappers.TransformObservation(e, lambda o: o[:2].astype(np.float32),\n                                                      gym.spaces.Box(-1.0, 1.0, (2,), np.float32))  # drop angular velocity\n                if stack > 1:\n                    e = gym.wrappers.FrameStackObservation(e, stack)\n                    e = gym.wrappers.FlattenObservation(e)\n                return e\n            return f\n        return GymVec(thunk, a.num_envs, a.seed)\n    if KIND == "mujoco":\n        name = a.env.split(":")[1]\n        pomdp = name.endswith("-pomdp")\n        name = name.replace("-pomdp", "")\n\n        def thunk(i):\n            def f():\n                e = gym.make(name)\n                if pomdp:   # keep joint positions only (first nq - 1 values), hide all velocities\n                    k = e.unwrapped.model.nq - 1\n                    e = gym.wrappers.TransformObservation(e, lambda o: o[:k].astype(np.float32),\n                                                          gym.spaces.Box(-np.inf, np.inf, (k,), np.float32))\n                if stack > 1:\n                    e = gym.wrappers.FrameStackObservation(e, stack)\n                    e = gym.wrappers.FlattenObservation(e)\n                return e\n            return f\n        return GymVec(thunk, a.num_envs, a.seed)\n    raise ValueError(a.env)\n\n\n# ================================================================== networks\ndef layer(m, std=math.sqrt(2)):\n    nn.init.orthogonal_(m.weight, std); nn.init.zeros_(m.bias)\n    return m\n\n\nclass Encoder(nn.Module):\n    def __init__(self, shape):\n        super().__init__()\n        if len(shape) == 3 and shape[-1] == 84:           # Atari\n            self.net = nn.Sequential(layer(nn.Conv2d(shape[0], 32, 8, 4)), nn.ReLU(), layer(nn.Conv2d(32, 64, 4, 2)), nn.ReLU(),\n                                     layer(nn.Conv2d(64, 64, 3, 1)), nn.ReLU(), nn.Flatten(), layer(nn.Linear(3136, 512)), nn.ReLU())\n            self.dim, self.scale = 512, 1 / 255.0\n        elif len(shape) == 3:                             # MinAtar 10x10\n            self.net = nn.Sequential(layer(nn.Conv2d(shape[0], 16, 3, 1)), nn.ReLU(), nn.Flatten(),\n                                     layer(nn.Linear(16 * 8 * 8, 128)), nn.ReLU())\n            self.dim, self.scale = 128, 1.0\n        else:                                             # vector\n            self.net = nn.Sequential(layer(nn.Linear(int(np.prod(shape)), 64)), nn.Tanh(), layer(nn.Linear(64, 64)), nn.Tanh())\n            self.dim, self.scale = 64, 1.0\n\n    def forward(self, x):\n        return self.net(x * self.scale)\n\n\nclass MLPCore(nn.Module):\n    def __init__(self, F_, d):\n        super().__init__(); self.fc = layer(nn.Linear(F_, d))\n\n    def init_state(self, n):\n        return {}\n\n    def step(self, x, st, done):\n        return torch.tanh(self.fc(x)) if CONT else F.relu(self.fc(x)), st\n\n\nclass LSTMCore(nn.Module):\n    def __init__(self, F_, d):\n        super().__init__(); self.cell = nn.LSTMCell(F_, d)\n\n    def init_state(self, n):\n        z = torch.zeros(n, self.cell.hidden_size, device=dev)\n        return {"h": z, "c": z.clone()}\n\n    def step(self, x, st, done):\n        keep = (1 - done)[:, None]\n        h, c = self.cell(x, (st["h"] * keep, st["c"] * keep))\n        return h, {"h": h, "c": c}\n\n\nclass DendraCore(nn.Module):\n    """DENDRA core run step by step: Hebbian synapses S, spiking dendritic E/I neurons, neuromodulators.\n    The state is cleared when a new episode starts (done = 1)."""\n\n    def __init__(self, F_, d, regions=2, heads=4):\n        super().__init__()\n        self.inp = layer(nn.Linear(F_, d), 1.0)\n        self.regions = nn.ModuleList([CorticalRegion(d, heads, int(1.5 * d), 4, 4) for _ in range(regions)])\n        self.nm = Neuromodulators(d, regions, heads)\n        self.L, self.h, self.d = regions, heads, d\n        self.rates = []\n\n    def init_state(self, n):\n        dh = self.d // self.h\n        return {"S": torch.zeros(self.L, n, self.h, dh, dh, device=dev), "m": torch.zeros(n, self.d, device=dev),\n                "t": torch.zeros(n, device=dev)}\n\n    def step(self, x, st, done):\n        keep = 1 - done\n        S = st["S"] * keep[None, :, None, None, None]\n        m, t = st["m"] * keep[:, None], st["t"] * keep\n        x = self.inp(x)\n        dec = self.nm.decay\n        m = dec * m + x\n        norm = (1 - dec ** (t + 1)) / (1 - dec)\n        z = 2 * torch.sigmoid(self.nm.proj(self.nm.norm(m / norm[:, None]))).view(-1, self.L, 1 + self.h)\n        gain, ach = z[..., :1], z[..., 1:]\n        h, newS = x[:, None], []\n        for l, reg in enumerate(self.regions):\n            h, Sl = reg.step(h, gain[:, None, l], ach[:, None, l], None, S[l])\n            newS.append(Sl)\n        self.rates.append(torch.stack([r.l5.last_rate for r in self.regions]))\n        return h[:, 0], {"S": torch.stack(newS), "m": m, "t": t + 1}\n\n    def seq(self, x, st, done):\n        """A whole segment of T steps at once (parallel over time) - same result as calling step() T times.\n        x [T, N, F]; done [T, N] (1 = this step starts a new episode -> clear the state before it)."""\n        Tn, N = done.shape\n        x = self.inp(x)                                                     # [T, N, d]\n        dec = self.nm.decay\n        m, t, ms, ts = st["m"], st["t"], [], []\n        for i in range(Tn):                                                  # neuromodulator baseline level (cheap)\n            keep = 1 - done[i]\n            m = dec * m * keep[:, None] + x[i]; t = t * keep\n            ms.append(m); ts.append(t); t = t + 1\n        M, Tt = torch.stack(ms), torch.stack(ts)\n        norm = (1 - dec ** (Tt + 1)) / (1 - dec)\n        z = 2 * torch.sigmoid(self.nm.proj(self.nm.norm(M / norm[..., None]))).view(Tn, N, self.L, 1 + self.h)\n        gain, ach = z[..., :1].transpose(0, 1), z[..., 1:].transpose(0, 1)  # [N, T, L, .]\n        seg = done.transpose(0, 1).cumsum(1)                                # [N, T] number of resets up to step t\n        same = (seg[:, :, None] == seg[:, None, :])                          # same episode\n        first = (seg == 0)                                                   # no reset yet -> still sees the old state\n        idx = torch.arange(Tn, device=x.device)\n        dist = (idx[:, None] - idx[None, :]).float()\n        h, newS, rates = x.transpose(0, 1), [], []\n        for l, reg in enumerate(self.regions):\n            a23 = reg.l23\n            xin = reg.l4a(h)\n            q, k, v, w = a23._proj(xin, ach[:, :, l])                       # [N,T,h,dh], w [N,T,h]\n            loglam = torch.log(a23.lam())                                    # [h]\n            D = torch.where(dist >= 0, torch.exp(loglam[:, None, None] * dist.clamp(min=0)), torch.zeros((), device=x.device))\n            D = D[None] * same[:, None]                                      # [N,h,T,T]\n            A = torch.einsum("bihd,bjhd->bhij", q, k) * D * w.permute(0, 2, 1)[:, :, None, :]\n            o = torch.einsum("bhij,bjhd->bihd", A, v)\n            S0 = st["S"][l]                                                  # [N,h,dh,dh]\n            dec_in = torch.exp(loglam[None, :] * (idx[:, None] + 1).float())                      # [T,h]\n            o = o + torch.einsum("bhde,bihe->bihd", S0, q) * (dec_in[None, :, :, None] * first[:, :, None, None])\n            last = seg[:, -1:]\n            dec_out = torch.exp(loglam[None, :] * (Tn - 1 - idx)[:, None].float())                # [T,h]\n            wv = v * (w * dec_out[None] * (seg == last)[:, :, None])[..., None]\n            S_end = (torch.exp(loglam * Tn)[None, :, None, None] * S0 * (seg[:, -1] == 0)[:, None, None, None]\n                     + torch.einsum("bjhd,bjhe->bhde", wv, k))\n            h = h + a23._finish(xin, o)\n            h = h + reg.l5(reg.l4b(h), gain[:, :, l])\n            newS.append(S_end); rates.append(reg.l5.last_rate)\n        self.rates.append(torch.stack(rates))\n        return h.transpose(0, 1), {"S": torch.stack(newS), "m": M[-1], "t": Tt[-1] + 1}\n\n\nclass Agent(nn.Module):\n    def __init__(self, obs_shape, n_act):\n        super().__init__()\n        self.enc = Encoder(obs_shape)\n        self.core = {"dendra": DendraCore, "lstm": LSTMCore, "mlp": MLPCore}[a.core](self.enc.dim, a.d)\n        self.pi = layer(nn.Linear(a.d, n_act), 0.01)\n        self.v = layer(nn.Linear(a.d, 1), 1.0)\n        if CONT:\n            self.log_std = nn.Parameter(torch.zeros(n_act))\n\n    def dist(self, h):\n        if CONT:\n            return torch.distributions.Normal(self.pi(h), self.log_std.exp())\n        return torch.distributions.Categorical(logits=self.pi(h))\n\n    def step(self, obs, st, done):\n        h, st = self.core.step(self.enc(obs), st, done)\n        return self.dist(h), self.v(h)[:, 0], st\n\n    def seq(self, obs, st, done):\n        """A whole segment [T, N, ...]: DENDRA runs in parallel over time; LSTM/MLP loop step by step."""\n        Tn, N = done.shape\n        f = self.enc(obs.flatten(0, 1)).view(Tn, N, -1)\n        if hasattr(self.core, "seq"):\n            h, st = self.core.seq(f, st, done)\n        else:\n            hs = []\n            for t in range(Tn):\n                ht, st = self.core.step(f[t], st, done[t]); hs.append(ht)\n            h = torch.stack(hs)\n        return self.dist(h), self.v(h)[..., 0], st\n\n\n# ================================================================== PPO\nclass RunningStd:   # reward normalization by the std of the discounted return (continuous control)\n    def __init__(self, n, gamma):\n        self.ret, self.g, self.cnt, self.mean, self.var = np.zeros(n), gamma, 1e-4, 0.0, 1.0\n\n    def __call__(self, r, done):\n        self.ret = self.ret * self.g + r\n        b_mean, b_var, b_cnt = self.ret.mean(), self.ret.var(), len(r)\n        delta, tot = b_mean - self.mean, self.cnt + b_cnt\n        self.mean += delta * b_cnt / tot\n        self.var = (self.var * self.cnt + b_var * b_cnt + delta ** 2 * self.cnt * b_cnt / tot) / tot\n        self.cnt = tot\n        self.ret[done > 0] = 0\n        return np.clip(r / math.sqrt(self.var + 1e-8), -10, 10)\n\n\nstack = 4 if (a.core == "mlp" and KIND in ("atari", "pendulum-pomdp", "mujoco")) else 1   # the memoryless MLP sees the last 4 frames\nenvs = make_vec(stack)\nagent = Agent(envs.obs_shape, envs.n_actions).to(dev)\nlr = a.lr or (3e-4 if CONT else 2.5e-4)\nopt = torch.optim.Adam(agent.parameters(), lr=lr, eps=1e-5)\ngamma, lam_gae = 0.99, 0.95\nclip = 0.2 if CONT else 0.1\nent_coef = 0.0 if CONT else 0.01\nN, Tn = a.num_envs, a.num_steps\na.epochs = a.epochs or (10 if CONT else 4)\nn_updates = a.total // (N * Tn)\nrnorm = RunningStd(N, gamma) if CONT else None\nprint(f"{a.env} | core {a.core} | {sum(p.numel() for p in agent.parameters())/1e3:.0f}K params | "\n      f"{\'continuous\' if CONT else \'discrete\'} ({envs.n_actions}) | {n_updates} updates", flush=True)\n\nobs = torch.tensor(envs.reset(), device=dev)\ndone = torch.zeros(N, device=dev)\nst = agent.core.init_state(N)\nep_ret, ep_len = np.zeros(N), np.zeros(N)\nfinished, curve, t0, gstep = [], [], time.time(), 0\nfor upd in range(1, n_updates + 1):\n    if a.max_minutes and time.time() - t0 > a.max_minutes * 60:\n        print(f"[{a.core}] time limit of {a.max_minutes} min reached, stopping at step {gstep}", flush=True)\n        break\n    for g_ in opt.param_groups:\n        g_["lr"] = lr * (1 - (upd - 1) / n_updates)\n    init_st = {k: v.clone() for k, v in st.items()}\n    B_obs = torch.zeros((Tn, N) + tuple(envs.obs_shape), device=dev)\n    B_act = torch.zeros((Tn, N, envs.n_actions) if CONT else (Tn, N), device=dev, dtype=torch.float32 if CONT else torch.long)\n    B_logp, B_rew, B_done, B_val = (torch.zeros(Tn, N, device=dev) for _ in range(4))\n    with torch.no_grad():\n        for t in range(Tn):\n            B_obs[t], B_done[t] = obs, done\n            dist, val, st = agent.step(obs, st, done)\n            act = dist.sample()\n            B_act[t], B_val[t] = act, val\n            B_logp[t] = dist.log_prob(act).sum(-1) if CONT else dist.log_prob(act)\n            o2, r, d = envs.step(act.cpu().numpy())\n            ep_ret += r; ep_len += 1\n            for i in np.nonzero(d)[0]:\n                finished.append((gstep, ep_ret[i])); ep_ret[i] = 0; ep_len[i] = 0\n            r_train = rnorm(r, d) if CONT else np.sign(r) if KIND == "atari" else r\n            B_rew[t] = torch.tensor(r_train, device=dev)\n            obs, done = torch.tensor(o2, device=dev), torch.tensor(d, device=dev)\n            gstep += N\n        if hasattr(agent.core, "rates"):\n            agent.core.rates.clear()\n        _, next_v, _ = agent.step(obs, {k: v.clone() for k, v in st.items()}, done)\n        adv = torch.zeros(Tn, N, device=dev); last = torch.zeros(N, device=dev)\n        for t in reversed(range(Tn)):\n            nonterm = 1 - (done if t == Tn - 1 else B_done[t + 1])\n            nv = next_v if t == Tn - 1 else B_val[t + 1]\n            delta = B_rew[t] + gamma * nv * nonterm - B_val[t]\n            last = delta + gamma * lam_gae * nonterm * last\n            adv[t] = last\n        ret = adv + B_val\n\n    env_per_mb = N // a.minibatches\n    for ep in range(a.epochs):\n        perm = torch.randperm(N)\n        for k in range(a.minibatches):\n            idx = perm[k * env_per_mb:(k + 1) * env_per_mb].to(dev)\n            s_ = {kk: (v[:, idx] if kk == "S" else v[idx]) for kk, v in init_st.items()}\n            if hasattr(agent.core, "rates"):\n                agent.core.rates.clear()\n            dist, val, _ = agent.seq(B_obs[:, idx], s_, B_done[:, idx])        # re-run the whole segment (BPTT)\n            logp = dist.log_prob(B_act[:, idx]); ent = dist.entropy()\n            if CONT:\n                logp, ent = logp.sum(-1), ent.sum(-1)\n            mb_adv = adv[:, idx]; mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)\n            ratio = (logp - B_logp[:, idx]).exp()\n            pg = torch.max(-mb_adv * ratio, -mb_adv * ratio.clamp(1 - clip, 1 + clip)).mean()\n            vloss = 0.5 * ((val - ret[:, idx]) ** 2).mean()\n            loss = pg + 0.5 * vloss - ent_coef * ent.mean()\n            if a.core == "dendra":\n                rates = torch.stack(agent.core.rates).mean(0)\n                loss = loss + ((rates - 0.15) ** 2).sum()   # homeostasis\n            opt.zero_grad(); loss.backward()\n            nn.utils.clip_grad_norm_(agent.parameters(), 0.5); opt.step()\n\n    if upd % max(1, n_updates // 40) == 0 or upd == n_updates:\n        recent = [r for _, r in finished[-50:]]\n        avg = float(np.mean(recent)) if recent else float("nan")\n        curve.append({"step": gstep, "return": avg, "time": time.time() - t0})\n        sps = gstep / (time.time() - t0)\n        print(f"[{a.core}] step {gstep:8d} | mean return (last 50 episodes) {avg:8.2f} | {sps:5.0f} steps/s | "\n              f"{(time.time()-t0)/60:5.1f} min", flush=True)\n\nfinal = [r for _, r in finished[-100:]]\nres = {"env": a.env, "core": a.core, "seed": a.seed, "total": gstep, "params": sum(p.numel() for p in agent.parameters()),\n       "final_return_mean": float(np.mean(final)) if final else None, "curve": curve,\n       "episodes": [(int(s), float(r)) for s, r in finished], "time_s": time.time() - t0}\nos.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)\njson.dump(res, open(a.out, "w"))\nprint(f"[{a.core}] DONE: mean return (last 100 episodes) {res[\'final_return_mean\']}, {res[\'time_s\']/60:.1f} min", flush=True)\n', 'experiments/rl/plot_ppo.py': '"""Plot PPO learning curves and print a summary table from runs/ppo_*.json."""\nimport glob, json, os, sys\nfrom collections import defaultdict\nimport numpy as np\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nsrc = sys.argv[1] if len(sys.argv) > 1 else "runs"\nout = sys.argv[2] if len(sys.argv) > 2 else "figures/rl_ppo.png"\nSTYLE = {"mlp": ("Memoryless (MLP/CNN)", "#9a9a9a", "--"), "lstm": ("LSTM", "#555555", ":"),\n         "lstm_big": ("Large LSTM (same #params)", "#222222", "-."), "dendra": ("DENDRA", "#2a6fdb", "-")}\nTITLE = {"minatar:breakout": "MinAtar Breakout (discrete)", "pendulum-pomdp": "Pendulum, velocity hidden (continuous)",\n         "atari:Breakout": "Atari Breakout (discrete)", "atari:Pong": "Atari Pong (discrete)",\n         "mujoco:HalfCheetah-v5-pomdp": "HalfCheetah, velocities hidden (continuous)", "mujoco:HalfCheetah-v5": "HalfCheetah (continuous)"}\nruns = defaultdict(lambda: defaultdict(list))\nfor f in glob.glob(f"{src}/ppo_*.json"):\n    r = json.load(open(f))\n    runs[r["env"]][r["core"]].append(r)\n\n\ndef smooth_curve(r, grid, win=100):\n    """Moving average of finished-episode returns, sampled on a grid of environment steps."""\n    ep = np.array(r["episodes"]) if r["episodes"] else np.zeros((0, 2))\n    out = []\n    for g in grid:\n        rr = ep[ep[:, 0] <= g][-win:, 1] if len(ep) else []\n        out.append(np.mean(rr) if len(rr) else np.nan)\n    return np.array(out)\n\n\nenvs = sorted(runs)\nfig, axes = plt.subplots(1, len(envs), figsize=(6.5 * len(envs), 4), squeeze=False)\nplt.rcParams.update({"font.size": 10})\nprint(f"{\'Environment\':38} {\'Core\':24} {\'Params\':>8} {\'Return, last 100 episodes\':>26} {\'Steps/s\':>10}")\nfor ax, env in zip(axes[0], envs):\n    total = max(r["total"] for rs in runs[env].values() for r in rs)\n    grid = np.linspace(total / 50, total, 50)\n    for core in ("mlp", "lstm", "lstm_big", "dendra"):\n        if core not in runs[env]: continue\n        lab, c, ls = STYLE[core]\n        curves = np.array([smooth_curve(r, grid) for r in runs[env][core]])\n        m = np.nanmean(curves, 0)\n        ax.plot(grid, m, ls, color=c, lw=1.8, label=f"{lab} ({len(curves)} seed{"s" if len(curves) > 1 else ""})")\n        if len(curves) > 1:\n            ax.fill_between(grid, np.nanmin(curves, 0), np.nanmax(curves, 0), color=c, alpha=0.12)\n        fin = [r["final_return_mean"] if r["final_return_mean"] is not None else np.nan for r in runs[env][core]]\n        sps = [r["total"] / r["time_s"] for r in runs[env][core]]\n        print(f"{env:38} {lab:24} {runs[env][core][0][\'params\']/1e3:7.0f}K {np.nanmean(fin) if not np.all(np.isnan(fin)) else np.nan:12.2f} ± {np.nanstd(fin) if not np.all(np.isnan(fin)) else np.nan:6.2f} {np.mean(sps):10.0f}")\n    ax.set_title(TITLE.get(env, env), loc="left", fontsize=11)\n    ax.set_xlabel("environment steps"); ax.set_ylabel("episode return (moving avg. of 100)")\n    ax.grid(alpha=0.25); ax.legend(fontsize=8, frameon=False)\n    for s_ in ("top", "right"): ax.spines[s_].set_visible(False)\nplt.tight_layout()\nos.makedirs(os.path.dirname(out) or ".", exist_ok=True)\nplt.savefig(out, dpi=150)\n'}
for path, src in FILES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(src)
print("Written:", list(FILES))

In [ ]:
# ==================== CONFIGURATION ====================
JOBS = [
    # (environment, total steps)
    ("atari:Breakout", 2_000_000),
    ("mujoco:HalfCheetah-v5-pomdp", 1_000_000),
]
CORES = ["mlp", "lstm", "dendra"]
SEEDS = [0]
MAX_MINUTES = 150          # time limit per run (minutes)
PARALLEL = 3               # number of runs at the same time (Kaggle has 4 CPU cores, 2 GPUs)
if os.environ.get("DENDRA_QUICK") == "1":   # very fast smoke test
    JOBS, MAX_MINUTES = [("atari:Breakout", 2048), ("mujoco:HalfCheetah-v5-pomdp", 2048)], 5
os.makedirs("runs", exist_ok=True)
plan = [(env, steps, core, seed) for env, steps in JOBS for seed in SEEDS for core in CORES]
print(len(plan), "runs:")
for j in plan: print("  ", j)

In [ ]:
import subprocess, time, torch
n_gpu = max(1, torch.cuda.device_count())
running, queue, done_jobs, k = [], list(plan), [], 0
t0 = time.time()

def name(env, core, seed):
    return f"runs/ppo_{env.replace(':', '_')}_{core}_s{seed}"

while queue or running:
    while queue and len(running) < PARALLEL:
        env, steps, core, seed = queue.pop(0)
        devc = f"cuda:{k % n_gpu}" if torch.cuda.is_available() else "cpu"; k += 1
        cmd = ["python", "experiments/rl/ppo_recurrent.py", "--env", env, "--core", core, "--seed", str(seed),
               "--total", str(steps), "--device", devc, "--max_minutes", str(MAX_MINUTES), "--out", name(env, core, seed) + ".json"]
        log = open(name(env, core, seed) + ".log", "w")
        running.append((subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "NT": "1"}), env, core, seed))
        print(f"> start {env} | {core} | seed {seed} on {devc}", flush=True)
    time.sleep(60 if os.environ.get('DENDRA_QUICK') != '1' else 5)
    still = []
    for pr, env, core, seed in running:
        lines = open(name(env, core, seed) + ".log").read().strip().splitlines()
        last = lines[-1] if lines else ""
        if pr.poll() is None:
            still.append((pr, env, core, seed)); print(f"  {env:30} {core:6} | {last[:110]}", flush=True)
        else:
            print(f"done {env} | {core} | seed {seed} (exit code {pr.returncode}): {last[:120]}", flush=True)
    running = still
    print(f"--- {(time.time()-t0)/60:.0f} min ---", flush=True)

In [ ]:
!python experiments/rl/plot_ppo.py runs rl_ppo.png
from IPython.display import Image, display
display(Image("rl_ppo.png"))

## Tips

* **More games:** `("atari:Pong", 2_000_000)`, `("atari:SpaceInvaders", 2_000_000)`; other robots: `"mujoco:Hopper-v5-pomdp"`, `"mujoco:Walker2d-v5-pomdp"`; fully observed versions: drop the `-pomdp` suffix.
* **More seeds** (`SEEDS = [0, 1, 2]`) for firmer conclusions; RL is very noisy.
* **Reference:** standard PPO on Breakout usually reaches a few dozen points after ~2M steps (8M frames) and a few hundred after 10M steps.
* JSON results and logs are in `runs/`; the figure is `rl_ppo.png`.